In [ ]:
from safetensors.torch import load_file
import pandas as pd
import polars as pl
import torch
from tqdm import tqdm
import numpy as np

EXPERT_ID = 7
W_ID = 1
LAYER_ID = 16


if __name__ == "__main__":
    files = []
    for i in range(100):
        p = f"output_100/layer={LAYER_ID:02}/router.safetensors"
        files.append(p)


    row_ids = []
    titles = []
    scores = []


    for path in tqdm(files):
        tensors = []
        weights = load_file(path, device="mps")
        for k, v in weights.items():
            row_id, title = k.split(".", 1) # split left most ".":
            row_ids.append(row_id)
            titles.append(title)

            score = v[-1, EXPERT_ID]
            scores.append(score)
    
    scores_tensor = torch.tensor(scores)
    top_1000 = torch.topk(scores_tensor, 1000, dim=0, sorted=True)
    top_1000_indices_np = top_1000.indices.cpu().numpy()
    top_1000_scores_np = top_1000.values.cpu().numpy()

    titles_np = np.array(titles)
    top_1000_titles = titles_np[top_1000_indices_np]

    df = pd.DataFrame({
        "title": top_1000_titles,
        "score": top_1000_scores_np,
    })

    df.to_csv(f"router_statistics_layer{LAYER_ID}_expert{EXPERT_ID}_w{W_ID}.csv", index=False)

In [90]:
import pandas as pd

py_df = pd.read_csv('data/py_repos.csv', usecols=['rank', 'username/repo_name'])
all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

{'100.chubin/wttr.in',
 '11.josephmisiti/awesome-machine-learning',
 '12.public-apis/public-apis',
 '13.donnemartin/system-design-primer',
 '13.huggingface/transformers',
 '15.psf/requests',
 '16.521xueweihan/HelloGitHub',
 '18.scrapy/scrapy',
 '19.soimort/you-get',
 '20.minimaxir/big-list-of-naughty-strings',
 '21.ageitgey/face_recognition',
 '22.apache/superset',
 '23.TheAlgorithms/Python',
 '24.deepfakes/faceswap',
 '25.3b1b/manim',
 '25.jackfrued/Python-100-Days',
 '26.tiangolo/fastapi',
 '26.vinta/awesome-python',
 '27.ytdl-org/youtube-dl',
 '28.fighting41love/funNLP',
 '29.0voice/interview_internal_reference',
 '30.localstack/localstack',
 '31.isocpp/CppCoreGuidelines',
 '32.apachecn/AiLearning',
 '33.pandas-dev/pandas',
 '34.floodsung/Deep-Learning-Papers-Reading-Roadmap',
 '35.testerSunshine/12306',
 '36.faif/python-patterns',
 '37.google-research/bert',
 '38.getsentry/sentry',
 '39.CorentinJ/Real-Time-Voice-Cloning',
 '40.swisskyrepo/PayloadsAllTheThings',
 '41.willmcgugan/ric

In [ ]:
from safetensors.torch import load_file
import polars as pl
from pathlib import Path
import pandas as pd
# defaultdict
from collections import defaultdict
import torch




schema = [('layer', pl.Int8), ('expert', pl.Int8)]
for _, row in py_df.iterrows():
    repo = row["username/repo_name"]
    schema.append((f'{repo}.w1', pl.Float16))
    schema.append((f'{repo}.w3', pl.Float16))

df = pl.DataFrame(schema=schema)

for layer in range(32):
    layer_path = Path(f'output/py/layer={layer:02d}')
    for expert in range(8):
        repos_to_activations = {}
        expert_path = layer_path / f'expert={expert}'

        w1_path = expert_path / 'w=1.safetensors'
        w3_path = expert_path / 'w=3.safetensors'

        w1 = load_file(w1_path)
        w3 = load_file(w3_path)

        assert set(w1.keys()) == all_safetensor_keys, f"Repo missing in w=1 for expert {expert}"
        assert set(w3.keys()) == all_safetensor_keys, f"Repo missing in w=3 for expert {expert}"

        for _, row in py_df.iterrows():
            key = f'{row["rank"]}.{row["username/repo_name"]}'

            w1_activation = w1[key].cpu().to(torch.float16).numpy()
            assert w1_activation.shape == (14336,), f"Unexpected shape for {key} in w=1: {w1_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w1'] = w1_activation


            w3_activation = w3[key].cpu().to(torch.float16).numpy()
            assert w3_activation.shape == (14336,), f"Unexpected shape for {key} in w=3: {w3_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w3'] = w3_activation
        
        curr_expert_df = pl.DataFrame(repos_to_activations)
        curr_expert_df = curr_expert_df.with_columns(pl.lit(layer, dtype=pl.Int8).alias("layer"), pl.lit(expert, dtype=pl.Int8).alias("expert"))
        curr_expert_df = curr_expert_df.select(pl.col(df.columns))


    df = pl.concat([df, curr_expert_df], how="vertical")

In [118]:
df

layer,expert,public-apis/public-apis.w1,public-apis/public-apis.w3,donnemartin/system-design-primer.w1,donnemartin/system-design-primer.w3,TheAlgorithms/Python.w1,TheAlgorithms/Python.w3,jackfrued/Python-100-Days.w1,jackfrued/Python-100-Days.w3,vinta/awesome-python.w1,vinta/awesome-python.w3,ytdl-org/youtube-dl.w1,ytdl-org/youtube-dl.w3,tensorflow/models.w1,tensorflow/models.w3,nvbn/thefuck.w1,nvbn/thefuck.w3,django/django.w1,django/django.w3,pallets/flask.w1,pallets/flask.w3,keras-team/keras.w1,keras-team/keras.w3,httpie/httpie.w1,httpie/httpie.w3,scikit-learn/scikit-learn.w1,scikit-learn/scikit-learn.w3,ansible/ansible.w1,ansible/ansible.w3,shadowsocks/shadowsocks.w1,shadowsocks/shadowsocks.w3,python/cpython.w1,python/cpython.w3,home-assistant/core.w1,home-assistant/core.w3,josephmisiti/awesome-machine-learning.w1,…,magenta/magenta.w3,locustio/locust.w1,locustio/locust.w3,pytorch/examples.w1,pytorch/examples.w3,jumpserver/jumpserver.w1,jumpserver/jumpserver.w3,luong-komorebi/Awesome-Linux-Software.w1,luong-komorebi/Awesome-Linux-Software.w3,PaddlePaddle/Paddle.w1,PaddlePaddle/Paddle.w3,reddit-archive/reddit.w1,reddit-archive/reddit.w3,charlax/professional-programming.w1,charlax/professional-programming.w3,instillai/TensorFlow-Course.w1,instillai/TensorFlow-Course.w3,python-poetry/poetry.w1,python-poetry/poetry.w3,open-mmlab/mmdetection.w1,open-mmlab/mmdetection.w3,toml-lang/toml.w1,toml-lang/toml.w3,junyanz/pytorch-CycleGAN-and-pix2pix.w1,junyanz/pytorch-CycleGAN-and-pix2pix.w3,bokeh/bokeh.w1,bokeh/bokeh.w3,sanic-org/sanic.w1,sanic-org/sanic.w3,binux/pyspider.w1,binux/pyspider.w3,nginx-proxy/nginx-proxy.w1,nginx-proxy/nginx-proxy.w3,cookiecutter/cookiecutter.w1,cookiecutter/cookiecutter.w3,chubin/wttr.in.w1,chubin/wttr.in.w3
i8,i8,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,…,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16
7,0,0.3359375,0.212891,-0.21582,0.236328,-0.5,0.439453,-0.367188,0.098633,-0.605469,0.613281,-0.984375,0.511719,-0.863281,-0.049316,-0.640625,0.182617,-0.539062,-0.017944,0.18457,0.048828,-0.112793,0.519531,-0.251953,1.015625,-0.09375,0.093262,-0.233398,0.341797,-0.980469,0.636719,-0.71875,-0.53125,-0.542969,-0.137695,-0.171875,…,-0.083008,0.5078125,1.2109375,-0.617188,0.699219,-0.804688,-0.408203,-0.578125,0.326172,-0.84375,0.173828,-0.578125,-0.59375,-0.792969,0.140625,-0.400391,0.746094,0.180664,0.6953125,-0.202148,-0.320312,0.015991,0.425781,-0.855469,0.470703,-0.769531,-0.176758,-0.363281,0.613281,-0.310547,0.6171875,0.057129,0.447266,-0.443359,0.703125,-0.149414,-0.294922
7,0,-0.478516,0.191406,0.063477,-0.077148,-0.683594,-0.326172,0.416016,0.490234,-0.176758,-0.269531,-0.345703,-0.136719,0.349609,-0.222656,0.087402,0.071777,-0.061768,0.136719,0.183594,0.018066,-0.011902,-0.038086,-0.01355,0.064453,-0.332031,0.086914,-0.122559,-0.220703,0.408203,0.675781,-0.824219,0.175781,0.519531,-0.464844,0.005676,…,0.203125,0.326172,-0.245117,-0.365234,0.9765625,0.345703,-0.023193,-0.285156,0.376953,-0.03418,-0.183594,-0.425781,-0.21582,-0.507812,-0.574219,0.671875,0.7734375,-0.119629,0.172852,-0.212891,0.341797,0.07373,0.199219,-0.761719,0.388672,-0.71875,0.628906,-0.474609,0.605469,-0.166992,-0.211914,0.318359,0.839844,0.447266,-0.316406,-0.157227,-0.476562
7,0,-1.210938,-0.018921,-0.527344,0.326172,-0.195312,0.363281,-0.068848,-0.070312,-0.832031,-0.065918,-0.210938,0.02832,-0.714844,0.6484375,-0.515625,0.102539,-0.296875,0.032715,-0.9375,1.109375,-0.289062,-0.142578,0.176758,0.1171875,-0.195312,0.671875,0.07666,0.347656,-0.195312,-0.031982,-0.542969,0.6953125,-0.496094,0.337891,-0.644531,…,-0.289062,-1.234375,0.088379,-0.474609,-0.335938,-0.445312,1.0078125,-0.3125,-0.396484,-0.84375,0.730469,-0.722656,-0.273438,-0.122559,0.582031,-0.542969,0.300781,-0.228516,-0.166992,-0.031982,0.339844,0.233398,0.016357,-0.155273,

In [99]:
schema = [(row["username/repo_name"], pl.List(pl.Float16)) for _, row in py_df.iterrows()]
df = pl.DataFrame(schema=schema)

public-apis/public-apis,donnemartin/system-design-primer,TheAlgorithms/Python,jackfrued/Python-100-Days,vinta/awesome-python,ytdl-org/youtube-dl,tensorflow/models,nvbn/thefuck,django/django,pallets/flask,keras-team/keras,httpie/httpie,scikit-learn/scikit-learn,ansible/ansible,shadowsocks/shadowsocks,python/cpython,home-assistant/core,josephmisiti/awesome-machine-learning,huggingface/transformers,psf/requests,521xueweihan/HelloGitHub,scrapy/scrapy,soimort/you-get,minimaxir/big-list-of-naughty-strings,ageitgey/face_recognition,apache/superset,deepfakes/faceswap,3b1b/manim,tiangolo/fastapi,fighting41love/funNLP,0voice/interview_internal_reference,localstack/localstack,isocpp/CppCoreGuidelines,apachecn/AiLearning,pandas-dev/pandas,floodsung/Deep-Learning-Papers-Reading-Roadmap,testerSunshine/12306,…,tornadoweb/tornado,eriklindernoren/ML-From-Scratch,google/python-fire,beurtschipper/Depix,keon/algorithms,wangzheng0822/algo,getredash/redash,tqdm/tqdm,nicolargo/glances,sebastianruder/NLP-progress,drduh/macOS-Security-and-Privacy-Guide,numpy/numpy,celery/celery,OWASP/CheatSheetSeries,facebookresearch/detectron2,microsoft/cascadia-code,deezer/spleeter,bitcoinbook/bitcoinbook,magenta/magenta,locustio/locust,pytorch/examples,jumpserver/jumpserver,luong-komorebi/Awesome-Linux-Software,PaddlePaddle/Paddle,reddit-archive/reddit,charlax/professional-programming,instillai/TensorFlow-Course,python-poetry/poetry,open-mmlab/mmdetection,toml-lang/toml,junyanz/pytorch-CycleGAN-and-pix2pix,bokeh/bokeh,sanic-org/sanic,binux/pyspider,nginx-proxy/nginx-proxy,cookiecutter/cookiecutter,chubin/wttr.in
list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],…,list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16],list[f16]
